# 03 Generate Clean CoT with Groq

Generate clean reasoning outputs for one prompt shard using Groq. This notebook keeps checkpoint/resume logic and writes failed records instead of crashing.


This cell installs and imports the Groq client dependencies plus utilities for JSONL, checkpointing, and progress display.

In [ ]:
%pip install -q groq pandas tqdm python-dotenv openai

from pathlib import Path
import json
import os
import re
import time

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI


This cell configures the Groq model, shard selection, checkpoint paths, final output path, and API key loading.

In [ ]:
# --- backend selection: "deepseek" (primary) or "openrouter" (backup) ---
BACKEND = "nvidia"  # switch to "openrouter" if DeepSeek direct goes down

MODEL_NAME = "deepseek-chat"          # deepseek-v4-flash maps to this on the direct API
OPENROUTER_MODEL = "deepseek/deepseek-r1:free"   # backup, do not change
NVIDIA_MODEL = "deepseek-ai/deepseek-v4-pro"

SHARD_ID = 1
NUM_SHARDS = 1          # we merged, single shard now
CHECKPOINT_INTERVAL = 10
MAX_SAMPLES = 1000
RETRY_FAILED = False
MAX_WORKERS = 3         # concurrent requests — safe with 2 keys

TEMPERATURE = 0.2
TOP_P = 0.9
MAX_TOKENS = 1500

PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
OUTPUT_ROOT = Path(r"E:\PFE_CoT_outputs")
PROMPT_DATA_PATH = OUTPUT_ROOT / "data" / "interim" / f"reasoning_prompts_shard{SHARD_ID}.jsonl"
CHECKPOINT_DIR = OUTPUT_ROOT / "data" / "checkpoint"
FINAL_DIR = OUTPUT_ROOT / "data" / "final"

SFT_CHECKPOINT_300_PATH = FINAL_DIR / "sft_ready_checkpoint_300.jsonl"

def model_slug(name):
    aliases = {
        "deepseek-chat": "deepseek-v4-flash",
        "deepseek/deepseek-r1:free": "deepseek-r1",
    }
    slug = aliases.get(name, re.sub(r"[^A-Za-z0-9]+", "", name).lower())
    return slug

MODEL_SAFE_NAME = model_slug(
    MODEL_NAME if BACKEND == "deepseek"
    else OPENROUTER_MODEL if BACKEND == "openrouter"
    else NVIDIA_MODEL
)
RUN_NAME = f"{BACKEND}_{MODEL_SAFE_NAME}_shard{SHARD_ID}"
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_checkpoint.jsonl"
FINAL_OUTPUT_PATH = FINAL_DIR / f"{RUN_NAME}.jsonl"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")

# DeepSeek direct — two keys, rotated per request
_ds_keys_raw = os.getenv("DEEPSEEK_API_KEYS") or os.getenv("DEEPSEEK_API_KEY", "")
DEEPSEEK_API_KEYS = [k.strip() for k in _ds_keys_raw.split(",") if k.strip()]

# OpenRouter backup — kept from before
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
# keep 2 OpenRouter keys as backup reserve
_or_keys_raw = os.getenv("OPENROUTER_API_KEYS") or OPENROUTER_API_KEY
OPENROUTER_API_KEYS = [k.strip() for k in _or_keys_raw.split(",") if k.strip()]


NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY", "")
if BACKEND == "nvidia" and not NVIDIA_API_KEY:
    raise RuntimeError("Set NVIDIA_API_KEY in .env before running.")

print("Nvidia key loaded:", bool(NVIDIA_API_KEY))

if BACKEND == "deepseek" and not DEEPSEEK_API_KEYS:
    raise RuntimeError("Set DEEPSEEK_API_KEY or DEEPSEEK_API_KEYS (comma-separated) in .env")
if BACKEND == "openrouter" and not OPENROUTER_API_KEYS:
    raise RuntimeError("Set OPENROUTER_API_KEY or OPENROUTER_API_KEYS in .env")

print("Backend:", BACKEND)
print("Run name:", RUN_NAME)
print("DeepSeek keys loaded:", len(DEEPSEEK_API_KEYS))
print("OpenRouter keys loaded:", len(OPENROUTER_API_KEYS))
print("Nvidia keys loaded:", len(NVIDIA_API_KEY))
print("Prompt shard:", PROMPT_DATA_PATH)
print("Checkpoint:", CHECKPOINT_PATH)
print("Final output:", FINAL_OUTPUT_PATH)

This cell loads the assigned prompt shard, verifies that `original_id` is present, and prints a quick category summary.

In [ ]:
def load_jsonl(path):
    records = []
    if not path.exists():
        return records
    with open(path, "r", encoding="utf-8") as input_handle:
        for line in input_handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


prompt_records = load_jsonl(PROMPT_DATA_PATH)
missing_original_id = [record for record in prompt_records if not record.get("original_id")]
if missing_original_id:
    raise KeyError("Prompt records must contain original_id. Re-run notebook 02 after updating the pipeline.")

if MAX_SAMPLES is not None:
    prompt_records = prompt_records[:MAX_SAMPLES]

print("Prompt records in this shard:", len(prompt_records))
if prompt_records:
    prompt_df = pd.DataFrame(prompt_records)
    print("Unique original_id values:", prompt_df["original_id"].nunique())
    print("Category distribution:")
    print(prompt_df["category"].value_counts(dropna=False).head(30))


This cell defines verdict parsing, label comparison, and output-record creation with up to three genuine model attempts per sample.

In [ ]:
MAX_GENERATION_ATTEMPTS = 1


def clean_label(value):
    text = "unknown" if value is None else str(value).strip()
    return text if text else "unknown"


def normalize_for_compare(value):
    text = clean_label(value).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" .;:,_`*")


def parse_verdict(text):
    if not isinstance(text, str) or text == "failed_generation":
        return "failed_generation" if text == "failed_generation" else "unknown"
    match = re.search(r"(?im)^\s*Verdict\s*:\s*(?:\n\s*)?([^\r\n]+)", text)
    if match:
        return match.group(1).strip().strip(" .;:,_`*")
    return "unknown"


def make_output_record(prompt_record):
    sample_id = int(prompt_record["sample_id"])
    original_id = str(prompt_record["original_id"])
    true_label = clean_label(prompt_record.get("true_label", "unknown"))
    expected = normalize_for_compare(true_label)

    for attempt_number in range(1, MAX_GENERATION_ATTEMPTS + 1):
        try:
            reasoning_text = generate_with_groq(prompt_record)
            parsed_verdict = parse_verdict(reasoning_text)
        except Exception:
            reasoning_text = "failed_generation"
            parsed_verdict = "exception"

        parsed_normalized = normalize_for_compare(parsed_verdict)
        if parsed_normalized == expected:
            print(f"[OK] sample_id={sample_id} original_id={original_id} attempt={attempt_number} verdict={parsed_verdict}")
            return {
                "sample_id": sample_id,
                "original_id": original_id,
                "backend": BACKEND,
                "model": MODEL_NAME,
                "shard_id": SHARD_ID,
                "source_dataset": prompt_record.get("source_dataset", "CICMalDroid2020"),
                "prompt": prompt_record.get("prompt", ""),
                "system_prompt": prompt_record.get("system_prompt", ""),
                "user_prompt": prompt_record.get("user_prompt", prompt_record.get("prompt", "")),
                "raw_generation": reasoning_text,
                "reasoning": reasoning_text,
                "prediction": parsed_verdict,
                "agreement": "OK",
            }

        if parsed_normalized in {"", "unknown", "exception", "failed_generation"}:
            print(f"[RETRY] sample_id={sample_id} original_id={original_id} attempt={attempt_number} no parseable verdict expected={true_label}")
        else:
            print(f"[MISMATCH] sample_id={sample_id} original_id={original_id} attempt={attempt_number} predicted={parsed_verdict} expected={true_label}")

    print(f"[MISMATCH] sample_id={sample_id} original_id={original_id} predicted={parsed_verdict} expected={true_label}")
    return {
        "sample_id": sample_id,
        "original_id": original_id,
        "backend": BACKEND,
        "model": MODEL_NAME,
        "shard_id": SHARD_ID,
        "source_dataset": prompt_record.get("source_dataset", "CICMalDroid2020"),
        "prompt": prompt_record.get("prompt", ""),
        "system_prompt": prompt_record.get("system_prompt", ""),
        "user_prompt": prompt_record.get("user_prompt", prompt_record.get("prompt", "")),
        "raw_generation": reasoning_text,
        "reasoning": reasoning_text,
        "prediction": parsed_verdict,
        "agreement": "MISMATCH",
    }


def make_failed_record(prompt_record, error):
    return {
        "sample_id": int(prompt_record["sample_id"]),
        "original_id": str(prompt_record["original_id"]),
        "backend": BACKEND,
        "model": MODEL_NAME,
        "shard_id": SHARD_ID,
        "source_dataset": prompt_record.get("source_dataset", "CICMalDroid2020"),
        "prompt": prompt_record.get("prompt", ""),
        "system_prompt": prompt_record.get("system_prompt", ""),
        "user_prompt": prompt_record.get("user_prompt", prompt_record.get("prompt", "")),
        "raw_generation": "failed_generation",
        "reasoning": "failed_generation",
        "prediction": "failed_generation",
        "error_message": str(error),
        "agreement": "FAILED",
    }


This cell defines checkpoint and final-output helpers keyed by `original_id` so resume logic does not rely on temporary `sample_id` values.

In [ ]:
def record_key(record):
    if record.get("original_id"):
        return str(record["original_id"])
    return f"missing_original_id::sample_{record['sample_id']}"


def records_by_original_id(paths):
    completed = {}
    for path in paths:
        for record in load_jsonl(path):
            completed[record_key(record)] = record
    return completed


def save_checkpoint(completed):
    temp_path = CHECKPOINT_PATH.with_suffix(CHECKPOINT_PATH.suffix + ".tmp")
    ordered_records = sorted(completed.values(), key=lambda record: (str(record.get("original_id", "")), int(record["sample_id"])))
    with open(temp_path, "w", encoding="utf-8") as output_handle:
        for record in ordered_records:
            output_handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    temp_path.replace(CHECKPOINT_PATH)


def save_final_output(completed):
    temp_path = FINAL_OUTPUT_PATH.with_suffix(FINAL_OUTPUT_PATH.suffix + ".tmp")
    with open(temp_path, "w", encoding="utf-8") as output_handle:
        for prompt_record in prompt_records:
            sample_key = str(prompt_record["original_id"])
            if sample_key in completed:
                output_handle.write(json.dumps(completed[sample_key], ensure_ascii=False) + "\n")
    temp_path.replace(FINAL_OUTPUT_PATH)


def write_jsonl(path, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


This cell creates the Groq client and defines the actual Groq chat-completion call used by the retry loop.

In [ ]:
current_key_index = [0]


MAX_GROQ_RATE_LIMIT_RETRIES = 5
INITIAL_GROQ_BACKOFF_SECONDS = 5
MAX_GROQ_BACKOFF_SECONDS = 60


_key_index = [0]   # shared counter for round-robin rotation

def _get_client():
    if BACKEND == "deepseek":
        keys = DEEPSEEK_API_KEYS
        base_url = "https://api.deepseek.com"
    elif BACKEND == "nvidia":
        keys = [os.getenv("NVIDIA_API_KEY", "")]
        base_url = "https://integrate.api.nvidia.com/v1"
    else:
        keys = OPENROUTER_API_KEYS
        base_url = "https://openrouter.ai/api/v1"
    key = keys[_key_index[0] % len(keys)]
    _key_index[0] += 1
    return OpenAI(base_url=base_url, api_key=key)

def generate_with_groq(prompt_record):
    active_model = (MODEL_NAME if BACKEND == "deepseek" 
                    else OPENROUTER_MODEL if BACKEND == "openrouter"
                    else NVIDIA_MODEL)
    system_prompt = prompt_record.get("system_prompt", "You are a malware-analysis reasoning assistant.")
    user_prompt = prompt_record.get("user_prompt", prompt_record.get("prompt", ""))
    backoff = 5
    for attempt in range(6):   # up to 5 retries
        try:
            client = _get_client()
            response = client.chat.completions.create(
                model=active_model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=TEMPERATURE,
                top_p=TOP_P,
                max_tokens=MAX_TOKENS,
            )
            raw = response.choices[0].message.content.strip()
            return re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
        except Exception as error:
            if attempt >= 5:
                raise
            print(f"[RATE_LIMIT/ERROR] {error} — sleeping {backoff}s before retry {attempt+1}")
            time.sleep(backoff)
            backoff = min(backoff * 2, 60)


MAIN GENERATION LOOP: 
This cell resumes completed generations, processes unfinished prompt records, and periodically saves checkpoint and final JSONL outputs.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

completed = records_by_original_id([FINAL_OUTPUT_PATH, CHECKPOINT_PATH])
print("Records already completed:", len(completed))

_completed_lock = threading.Lock()
_checkpoint_300_saved = [False]   # flag so we save it only once

def _save_300_checkpoint():
    if _checkpoint_300_saved[0]:
        return
    ok = [r for r in completed.values() if r.get("agreement") == "OK"]
    if len(ok) >= 300:
        write_jsonl(SFT_CHECKPOINT_300_PATH, ok[:300])
        _checkpoint_300_saved[0] = True
        print(f"\n[TRAINING READY] 300 OK samples reached — saved: {SFT_CHECKPOINT_300_PATH}\n")

def _process_one(prompt_record):
    sample_key = str(prompt_record["original_id"])
    with _completed_lock:
        already_done = sample_key in completed and completed[sample_key].get("prediction") != "failed_generation"
        failed_done = sample_key in completed and completed[sample_key].get("prediction") == "failed_generation"
    if already_done or (failed_done and not RETRY_FAILED):
        return None   # skip

    try:
        record = make_output_record(prompt_record)
    except Exception as error:
        print(f"Generation failed for original_id={sample_key}: {error}")
        record = make_failed_record(prompt_record, error)

    with _completed_lock:
        completed[sample_key] = record
    return record

to_process = [
    p for p in prompt_records
    if not (str(p["original_id"]) in completed and completed[str(p["original_id"])].get("prediction") != "failed_generation")
]
print(f"Remaining to process: {len(to_process)}")

processed_since_save = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(_process_one, p): p for p in to_process}
    for future in tqdm(as_completed(futures), total=len(futures)):
        result = future.result()
        if result is None:
            continue
        processed_since_save += 1
        if processed_since_save >= CHECKPOINT_INTERVAL:
            with _completed_lock:
                save_checkpoint(completed)
                save_final_output(completed)
                _save_300_checkpoint()
            processed_since_save = 0
            time.sleep(0.1)

# final save
with _completed_lock:
    save_checkpoint(completed)
    save_final_output(completed)
    _save_300_checkpoint()

print("Checkpoint saved:", CHECKPOINT_PATH)
print("Final output saved:", FINAL_OUTPUT_PATH)

This cell performs a quick final-output sanity check and previews parsed predictions from the generated file.

In [ ]:
# Quick sanity check.
final_records = load_jsonl(FINAL_OUTPUT_PATH)
print("Final records:", len(final_records))
if final_records:
    result_df = pd.DataFrame(final_records)
    print(result_df["prediction"].value_counts(dropna=False).head(30))
    display(result_df[["sample_id", "original_id", "backend", "model", "shard_id", "prediction"]].head())
